In [ ]:
import math
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot(title, label, x, result, expected):
    plt.plot(x, expected, label=f'Ground truth')
    plt.plot(x, result, label=f'Sequre {label}')
    plt.title(title)
    plt.xlabel('x')
    plt.ylabel('y')
    plt.legend()
    plt.grid(True)
    plt.show()

def mae(result, expected):
    return np.mean(np.abs(np.array(result) - np.array(expected)))

def mxae(result, expected):
    return np.max(np.abs(np.array(result) - np.array(expected)))

def by_interval(df):
    for interval, group in df.groupby('Interval'):
        display(group)

In [ ]:
""" Core methods """

# Until Codon Jupyter is fixed: Read the data from files
show_plots = False

dump_folder = "dump"
dump_files = [
    "decor_trig",
    "decor",
    "fourier",
    "cheby",
    "taylor"
    ]
nbit_fs = [64]
intervals_count = 1
cps = [0, 1]
exclude = ["sinh", "cosh", "tanh"]

df_data = {
    'Method': [],
    'Interval': [],
    'MAE': [],
    'MXAE': [],
    'Runtime (avg)': [],
    'Runtime (std)': [],
    'Partitions count': [],
    'Truncations count': []
    }

df_waves = {
    'x': [],
    'Method': [],
    'Expected': [],
    'Result': []
    }

for cp in cps:
    df_data[f'Bytes sent CP{cp}'] = []
    df_data[f'Requests sent CP{cp}'] = []

for dump_file in dump_files:
    for nbit_f in nbit_fs:
        for i in range(intervals_count):
            for cp in cps:
                try:
                    with open(f"{dump_folder}/{dump_file}_{i}_{nbit_f}_CP{cp}.p", "rb") as f:
                        data = pickle.load(f)
                        x = data['x']
                        interval = f"({round(min(x), 2)}, {round(max(x), 2)})"
                        for k, v in data.items():
                            if not k.endswith('_result'):
                                continue

                            skip = False
                            for exclude_item in exclude:
                                if exclude_item in k:
                                    skip = True
                                    break
                            
                            if skip:
                                continue

                            k = k.replace('_result', '')
                            expected = data[f"{k}_expected"]

                            runtime_avg = round(data[f"{k}_time"][0], 5)
                            runtime_std = round(data[f"{k}_time"][1], 5)
                            bytes_sent = int(data[f"{k}_bytes_sent"][0])
                            send_requests = int(data[f"{k}_send_requests"][0])
                            partitions_count = int(data[f"{k}_partitions_count"][0])
                            truncations_count = int(data[f"{k}_truncations_count"][0])
                            
                            if cp == 1:
                                df_data['Method'].append(f"{k}_{nbit_f}")
                                df_data['Interval'].append(interval)
                                df_data['MAE'].append(mae(v, expected))
                                df_data['MXAE'].append(mxae(v, expected))

                                df_data['Runtime (avg)'].append(runtime_avg)
                                df_data['Runtime (std)'].append(runtime_std)
                                df_data['Partitions count'].append(partitions_count)
                                df_data['Truncations count'].append(truncations_count)

                            df_data[f'Bytes sent CP{cp}'].append(bytes_sent)
                            df_data[f'Requests sent CP{cp}'].append(send_requests)

                            if cp == 1 and interval == "(-9.42, 9.42)":
                                df_waves['x'].append(x)
                                df_waves['Method'].append(f"{k}_{nbit_f}")
                                df_waves['Expected'].append(expected)
                                df_waves['Result'].append(v)

                            if show_plots and cp == 1:
                                plot(f"{k} {nbit_f} frac bits on {interval}", k, x, v, expected)
                except FileNotFoundError:
                    print(f"Could not find {dump_folder}/{dump_file}_{i}_{nbit_f}.p")

df = pd.DataFrame(df_data)

In [ ]:
if df.empty:
    raise ValueError("DataFrame is empty")

display_methods = [
    "sin_", "cos_", "tan_", "cot", "exp",
    "sigmoid", "sinh", "cosh", "tanh",
    "sqrt", "log", "mul_inv", "polynomial"]

for method in display_methods:
    display(by_interval(df[df['Method'].str.contains(method)].sort_values(by='MAE')))

In [ ]:
""" E2E apps """

# Until Codon Jupyter is fixed: Read the data from files
dump_files = [
    "log_reg",
    "dti"
    ]

df_data = {
    'Method': [],
    'Accuracy': [],
    'Runtime': [],
    'Bytes sent': [],
    'Requests sent': [],
    'Partitions count': [],
    'Truncations count': []
    }

for dump_file in dump_files:
    for nbit_f in nbit_fs:
        for cp in cps:
            try:
                with open(f"{dump_folder}/{dump_file}_{nbit_f}_CP{cp}.p", "rb") as f:
                    data = pickle.load(f)
                    for k, v in data.items():
                        if not k.endswith('_accuracy'):
                            continue
                        
                        k = k.replace('_accuracy', '')
                        accuracy = data[f"{k}_accuracy"][0]
                        runtime = round(data[f"{k}_time"][0], 5)
                        bytes_sent = int(data[f"{k}_bytes_sent"][0])
                        send_requests = int(data[f"{k}_send_requests"][0])
                        partitions_count = int(data[f"{k}_partitions_count"][0])
                        truncations_count = int(data[f"{k}_truncations_count"][0])
                        
                        df_data['Method'].append(f"{k}_{nbit_f}")
                        
                        df_data['Accuracy'].append(accuracy)
                        df_data['Runtime'].append(runtime)
                        df_data['Bytes sent'].append(bytes_sent)
                        df_data['Requests sent'].append(send_requests)
                        df_data['Partitions count'].append(partitions_count)
                        df_data['Truncations count'].append(truncations_count)
            except FileNotFoundError:
                print(f"Could not find {dump_folder}/{dump_file}_{nbit_f}_CP{cp}.p")

e2e_df = pd.DataFrame(df_data)

In [ ]:
# e2e_df[e2e_df['Method'].str.contains('binary')].sort_values(by='Accuracy', ascending=False)

In [ ]:
# e2e_df[e2e_df['Method'].str.contains('multinomial')].sort_values(by='Accuracy', ascending=False)

In [ ]:
# e2e_df[e2e_df['Method'].str.contains('dti')].sort_values(by='Accuracy', ascending=False)

In [ ]:
import re

def by_name(name, df):
    return df[df['Method'].str.startswith(name)]

def plot_accuracy_vs_perf(
        df, methods=["sin", "cos", "exp", "sigmoid"],
        metric="Runtime (avg)", intervals=["(-10.0, 10.0)"],
        approaches=["decor", "chebyshev", "fourier", "taylor"],
        markers=["o", "s", "^", "v"], colors=["tab:blue", "tab:orange", "tab:green", "tab:red"],
        remove_outliers=False, arangement="horizontal"):
    
    if arangement == "horizontal":
        fig, axs = plt.subplots(1, len(methods), figsize=(5 * len(methods), 5))
    elif arangement == "vertical":
        fig, axs = plt.subplots(len(methods), 1, figsize=(5, 5 * len(methods)))
    elif arangement == "square":
        n = int(np.ceil(np.sqrt(len(methods))))
        fig, axs = plt.subplots(n, n, figsize=(5 * n, 5 * n))
        axs = axs.flatten()
    else:
        raise ValueError(f"Unknown arangement: {arangement}")
    for idx, method in enumerate(methods):
        ax = axs[idx]

        filtered_df = df[df['Method'].str.contains(f"{method}_")]
        filtered_df = filtered_df[
            ~filtered_df['Method'].str.contains('_decor') &
            (~filtered_df['Method'].str.contains('_naive') |
             filtered_df['Method'].str.contains('taylor_'))
        ]
        filtered_df['Method'] = filtered_df['Method'].str.replace(f"{method}_", "", regex=False)
        
        if remove_outliers:
            filtered_df = filtered_df[filtered_df['MAE'] < 0.5]
        
        # Convert MAE to accuracy
        filtered_df['MAE'] = filtered_df['MAE'].apply(lambda x: -math.log2(x))
        
        for interval, group in filtered_df.groupby('Interval'):
            if interval not in intervals:
                print(f"Skipping interval {interval}")
                continue
            for approach, marker, color in zip(approaches, markers, colors):
                for nbit_f in nbit_fs:
                    filtered_group = group[
                        group['Method'].str.startswith(approach) &
                        group['Method'].str.endswith(f"{nbit_f}")].sort_values(by=metric)
                    if filtered_group.empty:
                        continue
                    if any(a in approach for a in ["chebyshev", "fourier", "taylor"]):
                        ax.plot(
                            filtered_group[metric], filtered_group["MAE"],
                            label=approach.capitalize(),
                            linestyle='--' if nbit_f == 64 else '-', marker=marker, linewidth=2,
                            color=color
                        )
                        if "chebyshev" in approach:
                            # Add label to each Chebyshev point
                            for x, y, name in zip(filtered_group[metric], filtered_group["MAE"], filtered_group['Method']):
                                # Try to parse first number from the left in approach string
                                match = re.search(r'\d+', name)
                                label = match.group(0) if match else name
                                ax.text(x, y, label, fontsize=8, ha='right', va='bottom')
                    else:
                        ax.scatter(
                            filtered_group[metric], filtered_group["MAE"],
                            label=approach.capitalize(), marker=marker, s=50,
                            color=color
                        )
            ax.set_title(f"{method} at {interval}")
            ax.set_xlabel(metric)
            ax.set_ylabel("Accuracy: 1 / MAE (log-scale)")
            ax.grid(True)
        if idx == len(methods) - 1:
            ax.legend(
                # title="Bordered: 64 bits\nNon-bordered: 32 bits",
                loc='lower right')
    
    plt.tight_layout()
    plt.show()

In [ ]:
plot_accuracy_vs_perf(df, metric="Runtime (avg)", approaches=["decor", "chebyshev", "taylor"])
plot_accuracy_vs_perf(df, metric="Bytes sent CP1", approaches=["decor", "chebyshev", "taylor"])

In [ ]:
# Plot sine, square wave, and sawtooth wave results and expected for degree 20, Chebyshev and Fourier

wave_types = ['sin', 'square', 'sawtooth']
degree = 50
methods = ['chebyshev', 'fourier']
colors = ['tab:green', 'tab:orange', 'tab:blue']

fig, axs = plt.subplots(len(wave_types), 1, figsize=(10, 4 * len(wave_types)), sharex=True)

for idx, wave in enumerate(wave_types):
    ax = axs[idx]
    mask = [
            (wave in m) and
            (f'_{degree}' in m)
            for m in df_waves['Method']
        ]
    if not any(mask):
        continue
    x = np.array(df_waves['x'])[mask][0]
    ground_truth = np.array(df_waves['Expected'])[mask][0]
    ax.plot(x, ground_truth, label=f'Ground truth', color=colors[0])
    for i, method in enumerate(methods):
        mask = [
            (wave in m) and
            (method in m) and
            (f'_{degree}' in m)
            for m in df_waves['Method']
        ]
        if not any(mask):
            continue
        result = np.array(df_waves['Result'])[mask][0]
        x = np.array(df_waves['x'])[mask][0]
        ax.plot(
            x, result,
            label=f'{"Decor " if "fourier" in method else ""}{method.capitalize()}',
            linestyle='--', color=colors[i + 1])
    ax.set_title(f'{wave.capitalize()} Wave (Degree {degree})')
    ax.grid(True)
    if idx == 0:
        ax.legend()


plt.xlabel('x')
plt.tight_layout()
plt.show()

In [ ]:
plot_accuracy_vs_perf(
        df, methods=["sin", "square", "sawtooth"],
        metric="Runtime (avg)", intervals=["(-9.42, 9.42)"],
        approaches=["fourier", "chebyshev"], arangement="vertical")
plot_accuracy_vs_perf(
        df, methods=["sin", "square", "sawtooth"],
        metric="Bytes sent CP1", intervals=["(-9.42, 9.42)"],
        approaches=["fourier", "chebyshev"], arangement="vertical")